# GeoNet — Kaggle 2×T4

Notebook d'entraînement prévu pour **Kaggle avec 2 GPU NVIDIA T4**.

Le notebook utilise **PyTorch DistributedDataParallel (DDP)** via :

```bash
torchrun --standalone --nproc_per_node=2
```

Architecture :

```text
Image 512×512
    ↓
EfficientNetV2-S pré-entraîné ImageNet
    ↓
features multi-échelles
    ↓
décodeur type U-Net
    ├── mask binaire 512×512
    └── heatmap des centres 256×256
```

Le dataset reste simplement :

```text
GeoNet (Kaggle dataset : `max778/geonet`)
├── images/
└── labels/
```

Format d'une pièce :

```text
0 cx cy p0x p0y ... p31x p31y
```

Les masks ne sont **pas stockés sur disque** : ils sont reconstruits en RAM depuis les 32 points.

### Configuration T4 recommandée

```text
Batch par GPU          = 4
Nombre de GPU          = 2
Gradient accumulation  = 2
Batch global effectif  = 4 × 2 × 2 = 16
AMP                    = FP16
```

Le modèle est beaucoup plus lourd que l'ancien PlankEye. Commence à `4/GPU`.
Si la VRAM le permet, teste ensuite `6/GPU`, puis `8/GPU` avec accumulation réduite.


## Datasets Kaggle utilisés

Dataset d'entraînement :

```text
Nom : GeoNet
Slug attendu : max778/geonet
Montage : /kaggle/input/geonet
```

Dataset de checkpoints :

```text
max778/checkpoints-geonet
Montage : /kaggle/input/checkpoints-geonet
```

Le notebook reprend automatiquement `last_geonet.pt` s'il est présent dans le dataset de checkpoints.


In [1]:
# ============================================================
# 1. Vérification des 2 GPU Kaggle
# ============================================================

import os
import sys
import subprocess
from pathlib import Path
import torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print()
subprocess.run(["nvidia-smi"], check=False)

print()
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU   :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB")

if torch.cuda.device_count() != 2:
    print()
    print("ATTENTION : ce notebook attend 2 GPU. Active l'accélérateur 2×T4 dans Kaggle.")


Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128

Fri Aug 21 06:15:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |              

In [8]:
# ============================================================
# CLONE GITHUB PRIVÉ - GeoNet / PlankEye
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import os
import shutil

GITHUB_USERNAME = "maaxxe"
GITHUB_REPO = "Train_Kaggle_plank_Detector"

REPO_DIR = Path(
    "/kaggle/working/Train_Kaggle_plank_Detector"
)

print("=" * 72)
print("CLONE GITHUB PRIVÉ")
print("=" * 72)

# ------------------------------------------------------------
# Récupération du token depuis les Secrets Kaggle
# ------------------------------------------------------------

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "Secret GITHUB_TOKEN introuvable.\n"
        "Ajoute-le dans Kaggle > Add-ons > Secrets "
        "et active-le pour ce notebook."
    )

repo_url = (
    f"https://{GITHUB_USERNAME}:{token}"
    f"@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
)

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

# ------------------------------------------------------------
# Supprime l'ancien dossier s'il existe
# pour repartir d'un clone propre
# ------------------------------------------------------------

if REPO_DIR.exists():
    print("Ancien repo détecté -> suppression...")
    shutil.rmtree(REPO_DIR)

# ------------------------------------------------------------
# Clone
# ------------------------------------------------------------

print("Clonage du repo...")

subprocess.run(
    [
        "git",
        "clone",
        repo_url,
        str(REPO_DIR),
    ],
    check=True,
    env=env,
)

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

print()
print("=" * 72)
print("CLONE TERMINÉ")
print("=" * 72)

print("Repo :", REPO_DIR)

GEONET_DIR = REPO_DIR / "GeoNet"
PLANK_EYE_DIR = REPO_DIR / "Plankeye"

print()
print("GeoNet   :", GEONET_DIR)
print("PlankEye :", PLANK_EYE_DIR)

# Vérification du modèle GeoNet
model_file = (
    GEONET_DIR
    / "model"
    / "multiforme_model.py"
)

print()
if model_file.exists():
    print("OK GeoNet model trouvé :")
    print(model_file)
else:
    print("ERREUR : multiforme_model.py introuvable.")

    print()
    print("Contenu du repo :")

    for p in REPO_DIR.rglob("*"):
        if p.is_file():
            print(" -", p.relative_to(REPO_DIR))

print()
print("=" * 72)

CLONE GITHUB PRIVÉ
Clonage du repo...


Cloning into '/kaggle/working/Train_Kaggle_plank_Detector'...



CLONE TERMINÉ
Repo : /kaggle/working/Train_Kaggle_plank_Detector

GeoNet   : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet
PlankEye : /kaggle/working/Train_Kaggle_plank_Detector/Plankeye

OK GeoNet model trouvé :
/kaggle/working/Train_Kaggle_plank_Detector/GeoNet/model/multiforme_model.py



In [9]:
from pathlib import Path

print("=" * 80)
print("CONTENU DE /kaggle/working")
print("=" * 80)

working = Path("/kaggle/working")

for p in sorted(working.rglob("*")):
    if p.is_file():
        try:
            rel = p.relative_to(working)
            print(rel)
        except Exception:
            print(p)

print()
print("=" * 80)
print("RECHERCHE DES FICHIERS MODEL PYTHON")
print("=" * 80)

for p in working.rglob("*.py"):
    if "model" in p.name.lower() or "geonet" in str(p).lower():
        print(p)

print()
print("=" * 80)
print("RECHERCHE multiforme_model.py")
print("=" * 80)

found = list(working.rglob("multiforme_model.py"))

if found:
    for p in found:
        print("TROUVÉ :", p)
else:
    print("AUCUN multiforme_model.py trouvé")

CONTENU DE /kaggle/working
.virtual_documents/__notebook_source__.ipynb
Train_Kaggle_plank_Detector/.git/HEAD
Train_Kaggle_plank_Detector/.git/config
Train_Kaggle_plank_Detector/.git/description
Train_Kaggle_plank_Detector/.git/hooks/applypatch-msg.sample
Train_Kaggle_plank_Detector/.git/hooks/commit-msg.sample
Train_Kaggle_plank_Detector/.git/hooks/fsmonitor-watchman.sample
Train_Kaggle_plank_Detector/.git/hooks/post-update.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-applypatch.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-commit.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-merge-commit.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-push.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-rebase.sample
Train_Kaggle_plank_Detector/.git/hooks/pre-receive.sample
Train_Kaggle_plank_Detector/.git/hooks/prepare-commit-msg.sample
Train_Kaggle_plank_Detector/.git/hooks/push-to-checkout.sample
Train_Kaggle_plank_Detector/.git/hooks/update.sample
Train_Kaggle_plank_Detector/.git

## 2. Localiser le projet et le dataset

Structure recommandée :

```text
/kaggle/working/GeoNet/
├── dataset_shapes/
├── model/
│   ├── __init__.py
│   └── multiforme_model.py
└── ...
```

Le notebook cherche en priorité le dataset **GeoNet** dans `/kaggle/input/geonet`.
Les checkpoints sont cherchés dans le dataset Kaggle **`max778/checkpoints-geonet`**, monté normalement sous `/kaggle/input/checkpoints-geonet`.


In [11]:
# ============================================================
# 2. Chemins GeoNet / Kaggle
# ============================================================

from pathlib import Path
import os
import sys

# ------------------------------------------------------------
# Datasets Kaggle
# ------------------------------------------------------------

GEONET_DATASET_SLUG = "max778/geonet"
GEONET_KAGGLE_DIR = Path("/kaggle/input/geonet")

CHECKPOINT_DATASET_SLUG = "max778/checkpoints-geonet"
CHECKPOINT_KAGGLE_DIR = Path("/kaggle/input/checkpoints-geonet")


# ============================================================
# Recherche du projet GeoNet
# ============================================================

def find_project_root():
    candidates = [
        Path("/kaggle/working/Train_Kaggle_plank_Detector/GeoNet"),
        Path("/kaggle/working/GeoNet"),
        Path.cwd(),
    ]

    # 1. Chemins connus
    for p in candidates:
        model_file = p / "model" / "multiforme_model.py"

        if model_file.is_file():
            print("Projet GeoNet trouvé :", p.resolve())
            return p.resolve()

    # 2. Recherche automatique
    kaggle_working = Path("/kaggle/working")

    if kaggle_working.exists():
        for model_file in kaggle_working.rglob("multiforme_model.py"):

            if model_file.parent.name != "model":
                continue

            project = model_file.parent.parent

            print(
                "Projet GeoNet trouvé automatiquement :",
                project.resolve()
            )

            return project.resolve()

    raise FileNotFoundError(
        "\nProjet GeoNet introuvable.\n\n"
        "Fichier recherché :\n"
        "  model/multiforme_model.py\n"
    )

# ============================================================
# Recherche dataset entraînement
# ============================================================

def find_dataset(project_root):

    # --------------------------------------------------------
    # 1. Dataset Kaggle officiel GeoNet
    # --------------------------------------------------------

    if (
        (GEONET_KAGGLE_DIR / "images").is_dir()
        and
        (GEONET_KAGGLE_DIR / "labels").is_dir()
    ):
        return GEONET_KAGGLE_DIR.resolve()

    # --------------------------------------------------------
    # 2. Dataset éventuellement présent dans le projet
    # --------------------------------------------------------

    local_data = project_root / "dataset_shapes"

    if (
        (local_data / "images").is_dir()
        and
        (local_data / "labels").is_dir()
    ):
        return local_data.resolve()

    # --------------------------------------------------------
    # 3. Recherche automatique dans /kaggle/input
    # --------------------------------------------------------

    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():

        candidates = []

        for images_dir in kaggle_input.rglob("images"):

            parent = images_dir.parent

            if (parent / "labels").is_dir():
                candidates.append(parent)

        # Priorité à GeoNet
        geonet_candidates = [
            p
            for p in candidates
            if "geonet" in str(p).lower()
            and "checkpoint" not in str(p).lower()
        ]

        if geonet_candidates:
            return geonet_candidates[0].resolve()

        if candidates:
            print(
                "ATTENTION : dataset GeoNet exact "
                "non trouvé."
            )

            print(
                "Utilisation du dataset détecté :",
                candidates[0]
            )

            return candidates[0].resolve()

    raise FileNotFoundError(
        "\nDataset GeoNet introuvable.\n\n"
        "Attendu normalement ici :\n"
        "  /kaggle/input/geonet/images\n"
        "  /kaggle/input/geonet/labels\n"
    )


# ============================================================
# Recherche dataset checkpoints
# ============================================================

def find_checkpoint_dataset():

    # Chemin attendu
    if CHECKPOINT_KAGGLE_DIR.is_dir():
        return CHECKPOINT_KAGGLE_DIR.resolve()

    # Recherche automatique
    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():

        for p in kaggle_input.iterdir():

            if (
                p.is_dir()
                and "checkpoint" in p.name.lower()
                and "geonet" in p.name.lower()
            ):
                return p.resolve()

    return None


# ============================================================
# Initialisation
# ============================================================

PROJECT_ROOT = find_project_root()

DATA_DIR = find_dataset(PROJECT_ROOT)

CHECKPOINT_DATA_DIR = find_checkpoint_dataset()


# ============================================================
# Configuration Python
# ============================================================

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# ============================================================
# Affichage
# ============================================================

print("=" * 72)
print("CONFIGURATION GEONET")
print("=" * 72)

print("PROJECT_ROOT          :", PROJECT_ROOT)
print("DATA_DIR              :", DATA_DIR)

print(
    "Dataset Kaggle attendu:",
    GEONET_DATASET_SLUG
)

print(
    "Checkpoint dataset    :",
    CHECKPOINT_DATASET_SLUG
)

print(
    "CHECKPOINT_DATA_DIR   :",
    CHECKPOINT_DATA_DIR
)

print("=" * 72)


# ============================================================
# Vérification dataset
# ============================================================

images = [
    p
    for p in (DATA_DIR / "images").iterdir()
    if p.suffix.lower()
    in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp",
    }
]

labels = list(
    (DATA_DIR / "labels").glob("*.txt")
)

print("Images :", len(images))
print("Labels :", len(labels))


# ============================================================
# Vérification checkpoints
# ============================================================

if CHECKPOINT_DATA_DIR is not None:

    checkpoints = list(
        CHECKPOINT_DATA_DIR.rglob("*.pt")
    )

    print()
    print(
        "Checkpoints trouvés :",
        len(checkpoints)
    )

    for ckpt in checkpoints:
        print(" -", ckpt)

else:

    print()
    print(
        "ATTENTION : aucun dataset de "
        "checkpoints GeoNet détecté."
    )


# ============================================================
# Warning dataset
# ============================================================

if len(images) < 100:

    print()
    print(
        "ATTENTION : dataset très petit : "
        "utile pour tester, "
        "pas pour un vrai entraînement."
    )

Projet GeoNet trouvé : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet
CONFIGURATION GEONET
PROJECT_ROOT          : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet
DATA_DIR              : /kaggle/input/datasets/max778/geonet/dataset_shapes
Dataset Kaggle attendu: max778/geonet
Checkpoint dataset    : max778/checkpoints-geonet
CHECKPOINT_DATA_DIR   : None
Images : 5000
Labels : 5000

ATTENTION : aucun dataset de checkpoints GeoNet détecté.


## 3. Configuration 2×T4

`BATCH_PER_GPU` est le batch **sur chaque T4**.

Avec les valeurs par défaut :

```text
4 × 2 GPU × accumulation 2 = batch global effectif 16
```

Si la mémoire est confortable, teste ensuite :

```python
BATCH_PER_GPU = 6
```

puis éventuellement :

```python
BATCH_PER_GPU = 8
ACCUMULATION_STEPS = 1
```


In [12]:
# ============================================================
# 3. CONFIGURATION GeoNet
# ============================================================

IMG_SIZE = 640
EPOCHS = 120

BATCH_PER_GPU = 4
ACCUMULATION_STEPS = 2
WORKERS_PER_GPU = 2

VAL_RATIO = 0.15
SEED = 42

LR_HEAD = 1e-5
LR_BACKBONE = 5e-7
WEIGHT_DECAY = 1e-4

# Avec DDP on ne change pas requires_grad après construction.
# Le backbone reste dans DDP dès le début, mais son LR vaut 0
# pendant ces premières epochs.
FREEZE_BACKBONE_EPOCHS = 2

GRAD_CLIP = 1.0
PATIENCE = 25

PRETRAINED = True
USE_AMP = True

OUTPUT_DIR = PROJECT_ROOT / "runs" / "geonet_kaggle"

# ------------------------------------------------------------
# Reprise depuis le dataset Kaggle :
# max778/checkpoints-geonet
# ------------------------------------------------------------

AUTO_RESUME_FROM_KAGGLE_CHECKPOINTS = True


def find_resume_checkpoint(checkpoint_dir):
    if checkpoint_dir is None:
        return None

    # Priorité au dernier checkpoint GeoNet.
    preferred_names = [
        "last_geonet.pt",
        "best_geonet.pt",

        # Compatibilité avec un éventuel ancien nom.
        "last_multiforme.pt",
        "best_multiforme.pt",
    ]

    for filename in preferred_names:
        matches = list(
            checkpoint_dir.rglob(filename)
        )

        if matches:
            return matches[0]

    return None


if AUTO_RESUME_FROM_KAGGLE_CHECKPOINTS:
    RESUME = find_resume_checkpoint(
        CHECKPOINT_DATA_DIR
    )
else:
    RESUME = None

WORLD_SIZE = 2

GLOBAL_EFFECTIVE_BATCH = (
    BATCH_PER_GPU
    * WORLD_SIZE
    * ACCUMULATION_STEPS
)

print("Modèle                : GeoNet")
print("Resolution            :", IMG_SIZE)
print("Epochs                :", EPOCHS)
print("Batch / GPU           :", BATCH_PER_GPU)
print("GPU                   :", WORLD_SIZE)
print("Accumulation          :", ACCUMULATION_STEPS)
print("Batch global effectif :", GLOBAL_EFFECTIVE_BATCH)
print("Workers / GPU         :", WORKERS_PER_GPU)
print("AMP FP16              :", USE_AMP)
print("Output                :", OUTPUT_DIR)
print("Resume                :", RESUME)

if CHECKPOINT_DATA_DIR is None:
    print(
        "INFO : le dataset checkpoints-geonet n'est pas monté. "
        "L'entraînement démarrera sans reprise."
    )
elif RESUME is None:
    print(
        "INFO : checkpoints-geonet est monté, "
        "mais aucun last_geonet.pt / best_geonet.pt n'a été trouvé."
    )


Modèle                : GeoNet
Resolution            : 640
Epochs                : 120
Batch / GPU           : 4
GPU                   : 2
Accumulation          : 2
Batch global effectif : 16
Workers / GPU         : 2
AMP FP16              : True
Output                : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet/runs/geonet_kaggle
Resume                : None
INFO : le dataset checkpoints-geonet n'est pas monté. L'entraînement démarrera sans reprise.


## 4. Vérifier les labels

Avec 32 points, une ligne contient normalement :

```text
1 classe + 2 coordonnées centre + 64 coordonnées contour = 67 valeurs
```

Le dataloader reste flexible et acceptera aussi un futur dataset à 64 points.


In [13]:
# ============================================================
# 4. Vérification rapide des labels
# ============================================================

label_files = sorted((DATA_DIR / "labels").glob("*.txt"))
bad = []
empty = 0
objects = 0
point_counts = set()

for path in label_files:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        empty += 1
        continue

    for line_no, line in enumerate(text.splitlines(), start=1):
        vals = line.split()
        if len(vals) < 7 or (len(vals) - 3) % 2 != 0:
            bad.append((path.name, line_no, len(vals)))
            continue
        objects += 1
        point_counts.add((len(vals) - 3) // 2)

print("Labels          :", len(label_files))
print("Images vides    :", empty)
print("Objets          :", objects)
print("Nb points trouvé:", sorted(point_counts))

if bad:
    print("Exemples invalides :", bad[:10])
    raise RuntimeError("Labels invalides détectés.")

print("Labels : OK")


Labels          : 5000
Images vides    : 395
Objets          : 11208
Nb points trouvé: [32]
Labels : OK


## 5. Créer le script DDP

`torchrun` lance deux processus Python indépendants, un par T4. La cellule suivante crée donc :

```text
/kaggle/working/GeoNet/train_geonet_kaggle_ddp.py
```

Le script gère :

- backend NCCL ;
- un processus par GPU ;
- `DistributedSampler` ;
- AMP FP16 ;
- gradient accumulation ;
- `DDP.no_sync()` entre micro-batches ;
- réduction des métriques entre les 2 GPU ;
- sauvegarde uniquement par le `rank 0` ;
- checkpoints et reprise.


In [14]:
# ============================================================
# 5. Ecriture du script DDP GeoNet
# ============================================================

DDP_SCRIPT = '# -*- coding: utf-8 -*-\n"""\nGeoNet - entraînement DDP Kaggle 2xT4.\n\nLancé par :\n    torchrun --standalone --nproc_per_node=2 train_geonet_kaggle_ddp.py ...\n\nLe --batch-per-gpu est PAR GPU.\nBatch effectif global =\n    batch_per_gpu * nombre_GPU * accumulation_steps\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom contextlib import nullcontext\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, Sampler, Subset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom tqdm.auto import tqdm\n\nfrom model import (\n    IMAGENET_MEAN,\n    IMAGENET_STD,\n    build_model,\n    multiforme_loss,\n    segmentation_metrics,\n)\n\n\n# ============================================================\n# DDP\n# ============================================================\n\ndef setup_ddp():\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA indisponible : ce script DDP attend des GPU NVIDIA.")\n\n    local_rank = int(os.environ["LOCAL_RANK"])\n    rank = int(os.environ["RANK"])\n    world_size = int(os.environ["WORLD_SIZE"])\n\n    torch.cuda.set_device(local_rank)\n    dist.init_process_group(backend="nccl")\n\n    device = torch.device("cuda", local_rank)\n\n    return local_rank, rank, world_size, device\n\n\ndef cleanup_ddp():\n    if dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef is_main(rank: int) -> bool:\n    return rank == 0\n\n\ndef reduce_sums(values: Dict[str, float], device: torch.device) -> Dict[str, float]:\n    keys = list(values.keys())\n    tensor = torch.tensor(\n        [values[k] for k in keys],\n        dtype=torch.float64,\n        device=device,\n    )\n    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)\n    return {k: float(v) for k, v in zip(keys, tensor.cpu().tolist())}\n\n\n# ============================================================\n# Reproductibilité\n# ============================================================\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\n# ============================================================\n# Dataset\n# ============================================================\n\ndef draw_gaussian_max(\n    heatmap: np.ndarray,\n    cx: float,\n    cy: float,\n    sigma: float,\n) -> None:\n    h, w = heatmap.shape\n    radius = max(1, int(math.ceil(3.0 * sigma)))\n\n    x0 = max(0, int(math.floor(cx)) - radius)\n    x1 = min(w, int(math.ceil(cx)) + radius + 1)\n    y0 = max(0, int(math.floor(cy)) - radius)\n    y1 = min(h, int(math.ceil(cy)) + radius + 1)\n\n    if x0 >= x1 or y0 >= y1:\n        return\n\n    xs = np.arange(x0, x1, dtype=np.float32)\n    ys = np.arange(y0, y1, dtype=np.float32)\n    yy, xx = np.meshgrid(ys, xs, indexing="ij")\n\n    gaussian = np.exp(\n        -((xx - cx) ** 2 + (yy - cy) ** 2)\n        / (2.0 * sigma ** 2)\n    )\n\n    region = heatmap[y0:y1, x0:x1]\n    np.maximum(region, gaussian, out=region)\n\n    ix = int(round(cx))\n    iy = int(round(cy))\n    if 0 <= ix < w and 0 <= iy < h:\n        heatmap[iy, ix] = 1.0\n\n\nclass MultiFormeDataset(Dataset):\n    def __init__(self, root: str | Path, img_size: int = 512):\n        self.root = Path(root)\n        self.image_dir = self.root / "images"\n        self.label_dir = self.root / "labels"\n\n        self.img_size = int(img_size)\n        self.center_size = self.img_size // 2\n\n        if not self.image_dir.is_dir():\n            raise FileNotFoundError(f"Dossier images introuvable : {self.image_dir}")\n        if not self.label_dir.is_dir():\n            raise FileNotFoundError(f"Dossier labels introuvable : {self.label_dir}")\n\n        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n        self.images = sorted(\n            p for p in self.image_dir.iterdir()\n            if p.suffix.lower() in extensions\n        )\n\n        if not self.images:\n            raise RuntimeError(f"Aucune image dans {self.image_dir}")\n\n        self.mean = np.asarray(\n            IMAGENET_MEAN,\n            dtype=np.float32,\n        ).reshape(1, 1, 3)\n\n        self.std = np.asarray(\n            IMAGENET_STD,\n            dtype=np.float32,\n        ).reshape(1, 1, 3)\n\n    def __len__(self):\n        return len(self.images)\n\n    def _read_objects(self, label_path: Path):\n        objects = []\n\n        if not label_path.exists():\n            return objects\n\n        text = label_path.read_text(encoding="utf-8").strip()\n        if not text:\n            return objects\n\n        for line_no, line in enumerate(text.splitlines(), start=1):\n            vals = line.strip().split()\n            if not vals:\n                continue\n\n            if len(vals) < 7:\n                raise ValueError(\n                    f"Label invalide {label_path}:{line_no}: "\n                    f"{len(vals)} valeurs."\n                )\n\n            cls = int(float(vals[0]))\n            cx = float(vals[1])\n            cy = float(vals[2])\n            coords = [float(v) for v in vals[3:]]\n\n            if len(coords) % 2 != 0:\n                raise ValueError(\n                    f"Nombre impair de coordonnées dans {label_path}:{line_no}"\n                )\n\n            points = np.asarray(coords, dtype=np.float32).reshape(-1, 2)\n            center = np.asarray([cx, cy], dtype=np.float32)\n\n            if points.shape[0] < 3:\n                continue\n\n            if not np.isfinite(points).all() or not np.isfinite(center).all():\n                raise ValueError(f"NaN/Inf dans {label_path}:{line_no}")\n\n            objects.append(\n                {\n                    "cls": cls,\n                    "center": np.clip(center, 0.0, 1.0),\n                    "points": np.clip(points, 0.0, 1.0),\n                }\n            )\n\n        return objects\n\n    def __getitem__(self, index: int):\n        image_path = self.images[index]\n        label_path = self.label_dir / f"{image_path.stem}.txt"\n\n        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n        if image_bgr is None:\n            raise RuntimeError(f"Impossible de lire {image_path}")\n\n        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n        image = cv2.resize(\n            image,\n            (self.img_size, self.img_size),\n            interpolation=cv2.INTER_AREA,\n        )\n\n        objects = self._read_objects(label_path)\n\n        # Pas de masks sur disque : reconstruction en RAM.\n        mask = np.zeros(\n            (self.img_size, self.img_size),\n            dtype=np.uint8,\n        )\n\n        center_heatmap = np.zeros(\n            (self.center_size, self.center_size),\n            dtype=np.float32,\n        )\n\n        for obj in objects:\n            pts_n = obj["points"]\n\n            pts_px = pts_n.copy()\n            pts_px[:, 0] *= self.img_size - 1\n            pts_px[:, 1] *= self.img_size - 1\n\n            poly = np.round(pts_px).astype(np.int32)\n            cv2.fillPoly(mask, [poly], 255)\n\n            cx_n, cy_n = obj["center"]\n            cx = float(cx_n) * (self.center_size - 1)\n            cy = float(cy_n) * (self.center_size - 1)\n\n            min_xy = pts_n.min(axis=0)\n            max_xy = pts_n.max(axis=0)\n            box_w = float(max_xy[0] - min_xy[0]) * self.center_size\n            box_h = float(max_xy[1] - min_xy[1]) * self.center_size\n\n            sigma = np.clip(\n                min(box_w, box_h) / 6.0,\n                1.5,\n                8.0,\n            )\n\n            draw_gaussian_max(\n                center_heatmap,\n                cx,\n                cy,\n                float(sigma),\n            )\n\n        image = image.astype(np.float32) / 255.0\n        image = (image - self.mean) / self.std\n        image = np.transpose(image, (2, 0, 1)).copy()\n\n        mask = (mask.astype(np.float32) / 255.0)[None, ...]\n        center_heatmap = center_heatmap[None, ...]\n\n        return {\n            "image": torch.from_numpy(image),\n            "mask": torch.from_numpy(mask),\n            "center": torch.from_numpy(center_heatmap),\n            "path": str(image_path),\n            "n_objects": len(objects),\n        }\n\n\n# ============================================================\n# Split et sampler validation sans duplication\n# ============================================================\n\ndef make_split(n: int, val_ratio: float, seed: int) -> Tuple[List[int], List[int]]:\n    if n < 2:\n        raise RuntimeError("Il faut au moins 2 images pour train/val.")\n\n    indices = list(range(n))\n    rng = random.Random(seed)\n    rng.shuffle(indices)\n\n    n_val = max(1, int(round(n * val_ratio)))\n    n_val = min(n_val, n - 1)\n\n    return indices[n_val:], indices[:n_val]\n\n\nclass DistributedEvalSampler(Sampler):\n    """Répartit la validation entre ranks sans dupliquer d\'échantillons."""\n\n    def __init__(self, dataset, num_replicas: int, rank: int):\n        self.dataset = dataset\n        self.num_replicas = num_replicas\n        self.rank = rank\n        self.indices = list(range(len(dataset)))[rank::num_replicas]\n\n    def __iter__(self):\n        return iter(self.indices)\n\n    def __len__(self):\n        return len(self.indices)\n\n\n# ============================================================\n# Epoch\n# ============================================================\n\ndef run_epoch(\n    model,\n    loader,\n    device,\n    rank: int,\n    optimizer=None,\n    scaler=None,\n    amp: bool = True,\n    grad_clip: float = 1.0,\n    accumulation_steps: int = 1,\n    train: bool = True,\n):\n    model.train(train)\n\n    sums = {\n        "loss": 0.0,\n        "bce": 0.0,\n        "dice_loss": 0.0,\n        "boundary": 0.0,\n        "center": 0.0,\n        "iou": 0.0,\n        "dice": 0.0,\n        "n_samples": 0.0,\n    }\n\n    if train:\n        optimizer.zero_grad(set_to_none=True)\n\n    iterator = loader\n    if is_main(rank):\n        iterator = tqdm(\n            loader,\n            leave=False,\n            desc="TRAIN" if train else "VAL",\n        )\n\n    n_batches = len(loader)\n\n    for step, batch in enumerate(iterator):\n        images = batch["image"].to(device, non_blocking=True)\n        masks = batch["mask"].to(device, non_blocking=True)\n        centers = batch["center"].to(device, non_blocking=True)\n\n        bs = images.shape[0]\n\n        should_step = (\n            (step + 1) % accumulation_steps == 0\n            or (step + 1) == n_batches\n        )\n\n        # Evite un all-reduce DDP sur chaque micro-batch lors de l\'accumulation.\n        sync_ctx = nullcontext()\n        if train and not should_step:\n            sync_ctx = model.no_sync()\n\n        with torch.set_grad_enabled(train):\n            with sync_ctx:\n                with torch.autocast(\n                    device_type="cuda",\n                    dtype=torch.float16,\n                    enabled=amp,\n                ):\n                    outputs = model(images)\n                    raw_loss, parts = multiforme_loss(\n                        outputs,\n                        masks,\n                        centers,\n                    )\n                    loss_for_backward = raw_loss / accumulation_steps\n\n                if train:\n                    if scaler is not None and scaler.is_enabled():\n                        scaler.scale(loss_for_backward).backward()\n                    else:\n                        loss_for_backward.backward()\n\n        if train and should_step:\n            if scaler is not None and scaler.is_enabled():\n                scaler.unscale_(optimizer)\n\n            torch.nn.utils.clip_grad_norm_(\n                model.parameters(),\n                grad_clip,\n            )\n\n            if scaler is not None and scaler.is_enabled():\n                scaler.step(optimizer)\n                scaler.update()\n            else:\n                optimizer.step()\n\n            optimizer.zero_grad(set_to_none=True)\n\n        metrics = segmentation_metrics(\n            outputs["mask_logits"],\n            masks,\n        )\n\n        sums["loss"] += float(raw_loss.detach()) * bs\n        sums["bce"] += parts["bce"] * bs\n        sums["dice_loss"] += parts["dice_loss"] * bs\n        sums["boundary"] += parts["boundary"] * bs\n        sums["center"] += parts["center"] * bs\n        sums["iou"] += metrics["iou"] * bs\n        sums["dice"] += metrics["dice"] * bs\n        sums["n_samples"] += bs\n\n        if is_main(rank):\n            local_n = max(1.0, sums["n_samples"])\n            iterator.set_postfix(\n                loss=f"{sums[\'loss\'] / local_n:.4f}",\n                iou=f"{sums[\'iou\'] / local_n:.3f}",\n                dice=f"{sums[\'dice\'] / local_n:.3f}",\n            )\n\n    # Additionne les statistiques des 2 GPU.\n    sums = reduce_sums(sums, device)\n    n = max(1.0, sums.pop("n_samples"))\n\n    return {\n        key: value / n\n        for key, value in sums.items()\n    }\n\n\n# ============================================================\n# Scheduler\n# ============================================================\n\ndef make_scheduler(\n    optimizer,\n    epochs: int,\n    freeze_backbone_epochs: int,\n    min_lr_ratio: float = 0.03,\n):\n    """\n    Cosine decay.\n\n    Pour le backbone :\n    - LR = 0 pendant freeze_backbone_epochs ;\n    - puis cosine decay.\n\n    Les paramètres gardent requires_grad=True afin que DDP les enregistre\n    correctement dès sa construction.\n    """\n\n    def head_lambda(epoch_idx: int):\n        progress = min(\n            max(epoch_idx / max(1, epochs - 1), 0.0),\n            1.0,\n        )\n        return min_lr_ratio + (\n            1.0 - min_lr_ratio\n        ) * 0.5 * (1.0 + math.cos(math.pi * progress))\n\n    def backbone_lambda(epoch_idx: int):\n        if epoch_idx < freeze_backbone_epochs:\n            return 0.0\n\n        active_epochs = max(1, epochs - freeze_backbone_epochs)\n        active_idx = epoch_idx - freeze_backbone_epochs\n        progress = min(\n            max(active_idx / max(1, active_epochs - 1), 0.0),\n            1.0,\n        )\n\n        return min_lr_ratio + (\n            1.0 - min_lr_ratio\n        ) * 0.5 * (1.0 + math.cos(math.pi * progress))\n\n    return torch.optim.lr_scheduler.LambdaLR(\n        optimizer,\n        lr_lambda=[\n            head_lambda,\n            backbone_lambda,\n        ],\n    )\n\n\n# ============================================================\n# Checkpoint\n# ============================================================\n\ndef save_checkpoint(\n    path: Path,\n    model: DDP,\n    optimizer,\n    scheduler,\n    epoch: int,\n    best_iou: float,\n    history: List[Dict],\n    args,\n):\n    checkpoint = {\n        "epoch": epoch,\n        "model": model.module.state_dict(),\n        "optimizer": optimizer.state_dict(),\n        "scheduler": scheduler.state_dict(),\n        "best_iou": best_iou,\n        "history": history,\n        "config": vars(args),\n    }\n    torch.save(checkpoint, path)\n\n\n# ============================================================\n# Main\n# ============================================================\n\ndef parse_args():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument("--data", type=str, required=True)\n    parser.add_argument("--output", type=str, default="runs/geonet_kaggle")\n\n    parser.add_argument("--img-size", type=int, default=512)\n    parser.add_argument("--epochs", type=int, default=120)\n\n    # IMPORTANT : batch PAR GPU.\n    parser.add_argument("--batch-per-gpu", type=int, default=4)\n    parser.add_argument("--accumulation", type=int, default=2)\n\n    # workers PAR GPU.\n    parser.add_argument("--workers", type=int, default=2)\n\n    parser.add_argument("--val-ratio", type=float, default=0.15)\n    parser.add_argument("--seed", type=int, default=42)\n\n    parser.add_argument("--lr-head", type=float, default=3e-4)\n    parser.add_argument("--lr-backbone", type=float, default=3e-5)\n    parser.add_argument("--weight-decay", type=float, default=1e-4)\n\n    parser.add_argument("--freeze-backbone-epochs", type=int, default=2)\n    parser.add_argument("--grad-clip", type=float, default=1.0)\n    parser.add_argument("--patience", type=int, default=25)\n\n    parser.add_argument("--resume", type=str, default="")\n    parser.add_argument("--no-pretrained", action="store_true")\n    parser.add_argument("--no-amp", action="store_true")\n\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n\n    local_rank, rank, world_size, device = setup_ddp()\n\n    torch.backends.cudnn.benchmark = True\n\n    # Split identique sur tous les ranks.\n    seed_everything(args.seed + rank)\n\n    output_dir = Path(args.output)\n    if is_main(rank):\n        output_dir.mkdir(parents=True, exist_ok=True)\n\n    dist.barrier()\n\n    dataset = MultiFormeDataset(\n        args.data,\n        img_size=args.img_size,\n    )\n\n    train_idx, val_idx = make_split(\n        len(dataset),\n        args.val_ratio,\n        args.seed,\n    )\n\n    train_set = Subset(dataset, train_idx)\n    val_set = Subset(dataset, val_idx)\n\n    train_sampler = DistributedSampler(\n        train_set,\n        num_replicas=world_size,\n        rank=rank,\n        shuffle=True,\n        seed=args.seed,\n        drop_last=False,\n    )\n\n    val_sampler = DistributedEvalSampler(\n        val_set,\n        num_replicas=world_size,\n        rank=rank,\n    )\n\n    train_loader = DataLoader(\n        train_set,\n        batch_size=args.batch_per_gpu,\n        sampler=train_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=False,\n    )\n\n    val_loader = DataLoader(\n        val_set,\n        batch_size=args.batch_per_gpu,\n        sampler=val_sampler,\n        num_workers=args.workers,\n        pin_memory=True,\n        persistent_workers=args.workers > 0,\n        drop_last=False,\n    )\n\n    if is_main(rank):\n        print("=" * 78)\n        print("MULTIFORMENET - KAGGLE DDP")\n        print("=" * 78)\n        print(f"GPU                 : {world_size}")\n        print(f"Train / Val         : {len(train_set)} / {len(val_set)}")\n        print(f"Image size          : {args.img_size}")\n        print(f"Batch / GPU         : {args.batch_per_gpu}")\n        print(f"Accumulation        : x{args.accumulation}")\n        print(\n            "Batch global effectif: "\n            f"{args.batch_per_gpu * world_size * args.accumulation}"\n        )\n        print(f"Workers / GPU       : {args.workers}")\n        print(f"AMP FP16            : {not args.no_amp}")\n        print(f"Pretrained          : {not args.no_pretrained}")\n        print("=" * 78)\n\n        if len(dataset) < 100:\n            print(\n                "ATTENTION : dataset très petit. "\n                "Le code fonctionnera mais ce n\'est pas suffisant pour entraîner le modèle."\n            )\n\n    # Rank 0 charge/télécharge d\'abord les poids ImageNet.\n    pretrained = not args.no_pretrained\n\n    if is_main(rank):\n        model = build_model(pretrained=pretrained)\n\n    dist.barrier()\n\n    if not is_main(rank):\n        model = build_model(pretrained=pretrained)\n\n    dist.barrier()\n\n    model = model.to(device)\n\n    # DDP doit voir tous les paramètres dès le départ.\n    model = DDP(\n        model,\n        device_ids=[local_rank],\n        output_device=local_rank,\n        broadcast_buffers=False,\n        find_unused_parameters=False,\n    )\n\n    head_params = [\n        p\n        for name, p in model.module.named_parameters()\n        if not name.startswith("encoder.")\n    ]\n\n    backbone_params = list(\n        model.module.encoder.parameters()\n    )\n\n    optimizer = torch.optim.AdamW(\n        [\n            {\n                "params": head_params,\n                "lr": args.lr_head,\n            },\n            {\n                "params": backbone_params,\n                "lr": args.lr_backbone,\n            },\n        ],\n        weight_decay=args.weight_decay,\n    )\n\n    scheduler = make_scheduler(\n        optimizer,\n        epochs=args.epochs,\n        freeze_backbone_epochs=args.freeze_backbone_epochs,\n    )\n\n    amp = not args.no_amp\n    scaler = torch.amp.GradScaler(\n        "cuda",\n        enabled=amp,\n    )\n\n    start_epoch = 1\n    best_iou = -1.0\n    history = []\n\n    if args.resume:\n        checkpoint = torch.load(\n            args.resume,\n            map_location=device,\n            weights_only=False,\n        )\n\n        model.module.load_state_dict(\n            checkpoint["model"]\n        )\n        optimizer.load_state_dict(\n            checkpoint["optimizer"]\n        )\n        scheduler.load_state_dict(\n            checkpoint["scheduler"]\n        )\n\n        start_epoch = int(checkpoint["epoch"]) + 1\n        best_iou = float(\n            checkpoint.get("best_iou", -1.0)\n        )\n        history = list(\n            checkpoint.get("history", [])\n        )\n\n        if is_main(rank):\n            print(\n                f"Reprise depuis {args.resume} "\n                f"-> epoch {start_epoch}"\n            )\n\n    dist.barrier()\n\n    epochs_without_improvement = 0\n    training_start = time.time()\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        train_sampler.set_epoch(epoch)\n\n        train_stats = run_epoch(\n            model=model,\n            loader=train_loader,\n            device=device,\n            rank=rank,\n            optimizer=optimizer,\n            scaler=scaler,\n            amp=amp,\n            grad_clip=args.grad_clip,\n            accumulation_steps=args.accumulation,\n            train=True,\n        )\n\n        with torch.no_grad():\n            val_stats = run_epoch(\n                model=model,\n                loader=val_loader,\n                device=device,\n                rank=rank,\n                optimizer=None,\n                scaler=None,\n                amp=amp,\n                grad_clip=args.grad_clip,\n                accumulation_steps=1,\n                train=False,\n            )\n\n        # Tous les ranks appellent le scheduler.\n        scheduler.step()\n\n        improved = val_stats["iou"] > best_iou\n\n        if improved:\n            best_iou = val_stats["iou"]\n            epochs_without_improvement = 0\n        else:\n            epochs_without_improvement += 1\n\n        if is_main(rank):\n            elapsed_min = (\n                time.time() - training_start\n            ) / 60.0\n\n            record = {\n                "epoch": epoch,\n                "train": train_stats,\n                "val": val_stats,\n                "lr_head": optimizer.param_groups[0]["lr"],\n                "lr_backbone": optimizer.param_groups[1]["lr"],\n            }\n            history.append(record)\n\n            print(\n                f"[{epoch:03d}/{args.epochs:03d}] "\n                f"train={train_stats[\'loss\']:.4f} "\n                f"IoU={train_stats[\'iou\']:.4f} "\n                f"| val={val_stats[\'loss\']:.4f} "\n                f"IoU={val_stats[\'iou\']:.4f} "\n                f"Dice={val_stats[\'dice\']:.4f} "\n                f"Center={val_stats[\'center\']:.4f} "\n                f"| LR={optimizer.param_groups[0][\'lr\']:.2e}/"\n                f"{optimizer.param_groups[1][\'lr\']:.2e} "\n                f"| {elapsed_min:.1f} min"\n            )\n\n            save_checkpoint(\n                output_dir / "last_geonet.pt",\n                model,\n                optimizer,\n                scheduler,\n                epoch,\n                best_iou,\n                history,\n                args,\n            )\n\n            if improved:\n                save_checkpoint(\n                    output_dir / "best_geonet.pt",\n                    model,\n                    optimizer,\n                    scheduler,\n                    epoch,\n                    best_iou,\n                    history,\n                    args,\n                )\n                print(\n                    f"  -> BEST : IoU={best_iou:.4f}"\n                )\n\n            (output_dir / "history.json").write_text(\n                json.dumps(history, indent=2),\n                encoding="utf-8",\n            )\n\n        # Tous les ranks prennent la même décision d\'arrêt.\n        stop_tensor = torch.tensor(\n            [\n                1\n                if epochs_without_improvement >= args.patience\n                else 0\n            ],\n            device=device,\n            dtype=torch.int32,\n        )\n        dist.broadcast(stop_tensor, src=0)\n\n        if int(stop_tensor.item()) == 1:\n            if is_main(rank):\n                print(\n                    f"Early stopping : {args.patience} epochs "\n                    "sans amélioration de l\'IoU."\n                )\n            break\n\n        dist.barrier()\n\n    if is_main(rank):\n        print("=" * 78)\n        print(f"Terminé. Best IoU = {best_iou:.4f}")\n        print(\n            "Best checkpoint :",\n            output_dir / "best_geonet.pt",\n        )\n        print("=" * 78)\n\n    cleanup_ddp()\n\n\nif __name__ == "__main__":\n    main()\n'

DDP_PATH = PROJECT_ROOT / 'train_geonet_kaggle_ddp.py'
DDP_PATH.write_text(DDP_SCRIPT, encoding='utf-8')

# Vérification syntaxique sans créer de __pycache__
import ast
ast.parse(DDP_SCRIPT)

print('Script créé :', DDP_PATH)
print('Syntaxe      : OK')


Script créé : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet/train_geonet_kaggle_ddp.py
Syntaxe      : OK


## 6. Lancer l'entraînement sur les 2 T4

Le `rank 0` affiche la progression et sauvegarde :

```text
runs/geonet_kaggle/
├── best_geonet.pt
├── last_geonet.pt
└── history.json
```

Le meilleur checkpoint est choisi selon l'**IoU de segmentation validation**.


In [21]:
# ============================================================
# LANCEMENT TORCHRUN - SORTIE EN DIRECT
# ============================================================

print()
print("=" * 72)
print("DÉBUT TORCHRUN")
print("=" * 72)
print()

process = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

all_lines = []

# Affichage en direct
for line in process.stdout:
    print(line, end="")
    all_lines.append(line)

returncode = process.wait()

print()
print("=" * 72)
print("FIN TORCHRUN")
print("=" * 72)
print("Code retour :", returncode)

# ============================================================
# Si erreur : afficher uniquement la partie importante
# ============================================================

if returncode != 0:

    print()
    print("=" * 72)
    print("DERNIÈRES 100 LIGNES")
    print("=" * 72)

    for line in all_lines[-100:]:
        print(line, end="")

    print()
    print("=" * 72)
    print("LIGNES CONTENANT ERROR / TRACEBACK / FAILED")
    print("=" * 72)

    keywords = [
        "error",
        "traceback",
        "failed",
        "exception",
        "modulenotfound",
        "importerror",
        "filenotfound",
        "runtimeerror",
        "typeerror",
        "valueerror",
        "keyerror",
        "attributeerror",
    ]

    found = False

    for i, line in enumerate(all_lines):
        lower = line.lower()

        if any(k in lower for k in keywords):
            found = True

            start = max(0, i - 3)
            end = min(len(all_lines), i + 10)

            print()
            print("-" * 72)

            for error_line in all_lines[start:end]:
                print(error_line, end="")

    if not found:
        print("Aucune ligne d'erreur explicite détectée.")

    raise RuntimeError(
        f"torchrun a échoué avec le code {returncode}"
    )

print()
print("✅ Entraînement terminé.")


DÉBUT TORCHRUN

[W821 06:24:25.357229200 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Traceback (most recent call last):
  File "/kaggle/working/Train_Kaggle_plank_Detector/GeoNet/train_geonet_kaggle_ddp.py", line 34, in <module>
    from model import (
ImportError: cannot import name 'IMAGENET_MEAN' from 'model' (unknown location)
Traceback (most recent call last):
  File "/kaggle/working/Train_Kaggle_plank_Detector/GeoNet/train_geonet_kaggle_ddp.py", line 34, in <module>
    from model import (
ImportError: cannot import name 'IMAGENET_MEAN' from 'model' (unknown location)
E0821 06:24:27.772000 171 torch/distributed/elastic/multiprocessing/api.py:984] failed (exitcode: 1) local_rank: 0 (pid: 177) of binary: /usr/bin/python3
Traceback (most recent call last):
  File "/usr/local/bin/torchrun", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/er

RuntimeError: torchrun a échoué avec le code 1

## 7. Vérifier la VRAM

Le nouveau modèle est plus lourd que PlankEye. Commence à `4/GPU`.

Si les deux T4 restent loin de leur limite mémoire, essaie `6/GPU`, puis `8/GPU`.
Un batch physique plus grand est généralement plus rapide qu'une accumulation de gradients.


In [16]:
# ============================================================
# 7. GPU / VRAM
# ============================================================

subprocess.run(["nvidia-smi"], check=False)


Fri Aug 21 06:21:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['nvidia-smi'], returncode=0)

## 8. Courbes après entraînement

Affichage de la loss, de l'IoU et du Dice.


In [17]:
# ============================================================
# 8. Graphiques
# ============================================================

import json
import matplotlib.pyplot as plt

history_path = OUTPUT_DIR / "history.json"

if not history_path.exists():
    print("Pas encore de history.json :", history_path)
else:
    history = json.loads(history_path.read_text(encoding="utf-8"))
    epochs_plot = [r["epoch"] for r in history]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["train"]["loss"] for r in history], label="Train loss")
    plt.plot(epochs_plot, [r["val"]["loss"] for r in history], label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("GeoNet — Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["train"]["iou"] for r in history], label="Train IoU")
    plt.plot(epochs_plot, [r["val"]["iou"] for r in history], label="Val IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.title("GeoNet — IoU")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_plot, [r["val"]["dice"] for r in history], label="Val Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title("GeoNet — Dice validation")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    best = max(history, key=lambda r: r["val"]["iou"])
    print(
        f"Best epoch : {best['epoch']} | "
        f"IoU={best['val']['iou']:.4f} | "
        f"Dice={best['val']['dice']:.4f}"
    )


Pas encore de history.json : /kaggle/working/Train_Kaggle_plank_Detector/GeoNet/runs/geonet_kaggle/history.json


## 9. Reprendre après interruption Kaggle

Le notebook est configuré pour utiliser automatiquement le dataset de checkpoints :

```text
max778/checkpoints-geonet
```

Kaggle le monte normalement ici :

```text
/kaggle/input/checkpoints-geonet/
```

Ordre de priorité pour la reprise :

```text
last_geonet.pt
best_geonet.pt
last_multiforme.pt   # compatibilité ancien nom
best_multiforme.pt   # compatibilité ancien nom
```

Si `last_geonet.pt` est présent, `RESUME` est renseigné automatiquement.

Les nouveaux checkpoints générés sont :

```text
runs/geonet_kaggle/
├── best_geonet.pt
├── last_geonet.pt
└── history.json
```
